# PROYECTO DE APLICACIÓN

## Historias clínicas

### Consulta histórica clínica 

In [5]:
import csv
import os
from datetime import datetime, date

ARCHIVO_CSV = "C:/Users/carte/OneDrive/Documentos/MASTER IA/PYTHON PROGRAMMING/PRACTICA/PROYECTO DE APLICACION/historias_clinicas.csv"
CAMPOS = ["id_paciente", "id_medico", "fecha_visita", "notas"]


def consulta_historia_clinica(id_paciente: str, ruta_csv: str = ARCHIVO_CSV):
    """
    Muestra por pantalla la fecha y las notas de cada visita del paciente.
    Cada visita en una línea. Si no hay visitas, no muestra nada.
    """
    if not os.path.exists(ruta_csv):
        # Si el fichero no existe, no hay visitas registradas
        return

    visitas = []
    with open(ruta_csv, mode="r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("id_paciente") == id_paciente:
                # Guardamos para poder ordenar por fecha
                fecha_str = (row.get("fecha_visita") or "").strip()
                notas = (row.get("notas") or "").strip()
                try:
                    fecha_dt = datetime.strptime(fecha_str, "%d-%m-%Y")
                except ValueError:
                    # Si alguna fecha viene mal, la mandamos al final
                    fecha_dt = datetime.max
                visitas.append((fecha_dt, fecha_str, notas))

    # Ordenar por fecha
    visitas.sort(key=lambda x: x[0])

    for _, fecha_str, notas in visitas:
        print(f"{fecha_str} - {notas}")



### Registra Historia Clínica 


In [32]:
def registra_historia_clinica(id_medico: str, ruta_csv: str = ARCHIVO_CSV):
    """
    Pide el DNI del paciente y las notas.
    Guarda automáticamente la fecha de hoy y añade el registro al CSV.
    """
    id_medico = str(id_medico).strip()

    # Validación de 4 dígitos
    if not (len(id_medico) == 4 and id_medico.isdigit()):
        raise ValueError("id_medico debe ser un número de 4 dígitos (ej: '1234').")

    id_paciente = input("Introduce el DNI del paciente: ").strip()
    notas = input("Introduce las notas de la visita: ").strip()

    fecha_hoy = date.today().strftime("%d-%m-%Y")

    # Si el fichero no existe, lo creamos con cabecera
    crear_cabecera = not os.path.exists(ruta_csv)

    with open(ruta_csv, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS)
        if crear_cabecera:
            writer.writeheader()

        writer.writerow({
            "id_paciente": id_paciente,
            "id_medico": id_medico,
            "fecha_visita": fecha_hoy,
            "notas": notas
        })

    print("Visita registrada correctamente.")


In [33]:
registra_historia_clinica(1234)

Introduce el DNI del paciente:  03938893M
Introduce las notas de la visita:  dolor de cabez


Visita registrada correctamente.


## Pacientes

### Registro de pacientes

In [11]:
pacientes = {
    '00000000A': ['Luis Martin', 34, '10-06-2024', '17-06-2024'],
    '11111111B': ['Maria Diaz', 41, '17-06-2024', None],
}

from datetime import date

def registra_paciente(pacientes: dict):
    """
    Registra un nuevo paciente en el diccionario.
    Devuelve el diccionario actualizado.
    """

    dni = input("Introduce el DNI del paciente: ").strip()
    nombre = input("Introduce el nombre del paciente: ").strip()

    # Validación de la edad
    while True:
        edad_input = input("Introduce la edad del paciente: ").strip()

        if edad_input.isdigit():
            edad = int(edad_input)
            if 0 <= edad <= 120:
                break

        print("Error: la edad debe ser un número entero entre 0 y 120.")

    fecha_ingreso = date.today().strftime("%d-%m-%Y")
    fecha_alta = None

    pacientes[dni] = [nombre, edad, fecha_ingreso, fecha_alta]

    return pacientes


In [13]:
pacientes = registra_paciente(pacientes)


Introduce el DNI del paciente:  03938893M
Introduce el nombre del paciente:  Carlos Moreno
Introduce la edad del paciente:  27


### Dar de alta pacientes

In [14]:
from datetime import date

def alta_paciente(pacientes: dict):
    """
    Da de alta a un paciente actualizando su fecha de alta.
    Devuelve el diccionario de pacientes.
    """

    dni = input("Introduce el DNI del paciente a dar de alta: ").strip()

    # Comprobar si el paciente existe
    if dni not in pacientes:
        print("Error: el paciente no existe.")
        return pacientes

    # Obtener la fecha de hoy
    fecha_alta = date.today().strftime("%d-%m-%Y")

    # Actualizar la fecha de alta (último elemento de la lista)
    pacientes[dni][3] = fecha_alta

    return pacientes


In [17]:
alta_paciente(pacientes)

Introduce el DNI del paciente a dar de alta:  03938893M


{'00000000A': ['Luis Martin', 34, '10-06-2024', '17-06-2024'],
 '11111111B': ['Maria Diaz', 41, '17-06-2024', None],
 '03938893M': ['Carlos Moreno', 27, '18-01-2026', '18-01-2026']}

### Consulta información paciente 


In [18]:
def consulta_info_paciente(pacientes: dict, dni: str):
    """
    Imprime la información de un paciente.
    """

    # Comprobar si el paciente existe
    if dni not in pacientes:
        print("Error: el paciente no existe.")
        return

    # Información del paciente
    nombre, edad, fecha_ingreso, fecha_alta = pacientes[dni]

    print(f"Nombre: {nombre}")
    print(f"Edad: {edad}")
    print(f"Fecha de ingreso: {fecha_ingreso}")

    if fecha_alta is not None:
        print(f"Fecha de alta: {fecha_alta}")
    else:
        print("Fecha de alta: el paciente aún no ha sido dado de alta")


In [23]:
consulta_info_paciente(pacientes, "03938893M")


Nombre: Carlos Moreno
Edad: 27
Fecha de ingreso: 18-01-2026
Fecha de alta: 18-01-2026


## Médico

In [35]:
medicos = {
    1234: 'IEP2024',
    5555: 'M4DR1D'
}

def gestion_acceso(medicos: dict):
    """
    Controla el acceso de médicos a la aplicación.
    Devuelve el identificador del médico si el acceso es correcto.
    """

    id_medico_input = input("Introduce el identificador del médico: ").strip()

    # Comprobamos que el identificador sea numérico
    if not id_medico_input.isdigit():
        raise ValueError("Identificador de médico no válido.")

    id_medico = int(id_medico_input)

    # Comprobamos que el médico exista
    if id_medico not in medicos:
        raise ValueError("El médico no existe.")

    # Se permiten hasta 3 intentos de contraseña
    for intento in range(1, 4):
        password = input("Introduce la contraseña: ").strip()

        if password == medicos[id_medico]:
            print("Acceso concedido.")
            return id_medico
        else:
            print(f"Contraseña incorrecta (intento {intento}/3).")

    # Si llega aquí, las 3 contraseñas han sido incorrectas
    raise ValueError("Acceso denegado: demasiados intentos fallidos.")


## Ejecución Principal

In [36]:
def mostrar_menu():
    print("\n")
    print("==========================")
    print(" BIENVENIDO AL HOSPITAL")
    print("==========================")
    print("1. Registrar paciente")
    print("2. Consultar información del paciente")
    print("3. Dar de alta paciente")
    print("4. Añadir historia clínica del paciente")
    print("5. Salir")
    print("\n")


# 1) LOGIN
id_medico = gestion_acceso(medicos)

# 2) BUCLE PRINCIPAL
while True:
    mostrar_menu()

    opcion_str = input("Introduzca una opción: ").strip()

    # Si no es un número se vuelve al menú
    if not opcion_str.isdigit():
        print("Opción no válida.")
        continue

    opcion = int(opcion_str)

    if opcion == 1:
        # 1. Registrar paciente
        pacientes = registra_paciente(pacientes)

    elif opcion == 2:
        # 2. Consultar información del paciente e historia clínica
        dni = input("Introduzca el DNI del paciente que quiere consultar: ").strip()
        consulta_info_paciente(pacientes, dni)
        print("\n--- HISTORIA CLÍNICA ---")
        consulta_historia_clinica(dni)

    elif opcion == 3:
        # 3. Dar de alta paciente
        pacientes = alta_paciente(pacientes)

    elif opcion == 4:
        # 4. Añadir historia clínica
        registra_historia_clinica(str(id_medico))

    elif opcion == 5:
        # FIN DEL PROGRAMA
        print("FIN DEL PROGRAMA")
        break

    else:
        # OPCION NO VALIDA
        print("Opción no válida.")


Introduce el identificador del médico:  1234
Introduce la contraseña:  IEP2024


Acceso concedido.


 BIENVENIDO AL HOSPITAL
1. Registrar paciente
2. Consultar información del paciente
3. Dar de alta paciente
4. Añadir historia clínica del paciente
5. Salir




Introduzca una opción:  2
Introduzca el DNI del paciente que quiere consultar:  03938893M


Nombre: Carlos Moreno
Edad: 27
Fecha de ingreso: 18-01-2026
Fecha de alta: 18-01-2026

--- HISTORIA CLÍNICA ---


 BIENVENIDO AL HOSPITAL
1. Registrar paciente
2. Consultar información del paciente
3. Dar de alta paciente
4. Añadir historia clínica del paciente
5. Salir




Introduzca una opción:  5


FIN DEL PROGRAMA
